# Meatadata Driven DLT Ingest

Runs dlt pipeline for the specified source.


In [ ]:
# Parameters
source_name = "my_source"
vault_url = "https://data-eng-key-vault.vault.azure.net/"
# Optional: pass source config directly as JSON string
# Secrets in config are Key Vault secret names, resolved at runtime
source_config_json = ""
github_pat_secret = "github-pat"  # LEGACY: secret name for Key Vault lookup
github_token = ""  # NEW: actual token passed directly (preferred)
run_dq = False        # When True, run dbt DQ tests after pipeline (even on failure)
dq_repo_url = ""      # Git repo containing .dq/ dbt project (required when run_dq=True)
dq_repo_branch = "main"



In [ ]:
# Install dependencies
!pip uninstall deltalake -y -q
!pip install "vd-dlt[pipeline,notion,notion-schema,salesforce,salesforce-schema]" 'deltalake>=0.18.0' -q


In [ ]:
# Setup source config
import json as _json

if source_config_json:
    print("Using source config from parameter")
    source_config = _json.loads(source_config_json)
else:
    raise ValueError("source_config_json is required - pass the source config as JSON")


In [ ]:
# Run pipeline with log capture
import os, logging, shutil
from datetime import datetime, timezone
from vd_dlt.runner import run_pipeline
from vd_dlt.fabric_utils import mount_lakehouse

# Get target lakehouse info from source config
target = source_config.get("target", {})
workspace_id = target.get("workspace_id")
lakehouse_id = target.get("lakehouse_id")

print(f"Target: {target.get('workspace')}/{target.get('lakehouse')}/{target.get('schema_name')}")

# Mount target lakehouse
destination_path = mount_lakehouse(workspace_id, lakehouse_id)

# Configure dlt logger to capture to file
run_ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
log_path = f'/tmp/dlt_{source_name}_{run_ts}.log'

dlt_logger = logging.getLogger('dlt')
dlt_logger.setLevel(logging.INFO)
file_handler = logging.FileHandler(log_path)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
dlt_logger.addHandler(file_handler)

os.environ['RUNTIME__LOG_LEVEL'] = 'INFO'

# Run the pipeline, always persist logs even on failure
try:
    results = run_pipeline(
        source_name=source_name,
        vault_url=vault_url,
        destination_path=destination_path,
        source_config=source_config,
    )
finally:
    # Flush and close log handler
    file_handler.flush()
    file_handler.close()
    dlt_logger.removeHandler(file_handler)

    # Copy log file to lakehouse Files section
    try:
        date_path = datetime.now(timezone.utc).strftime('%Y/%m/%d')
        user_id = source_config.get('user_id', 'unknown')
        files_base = destination_path.replace('/Tables', '/Files')
        log_dir = f'{files_base}/logs/dlt/{date_path}/{user_id}'
        log_dest = f'{log_dir}/{source_name}_{run_ts}.log'
        os.makedirs(log_dir, exist_ok=True)

        if os.path.exists(log_path):
            shutil.copy(log_path, log_dest)
            print(f"dlt logs copied to lakehouse: {log_dest}")
        else:
            print("Warning: dlt log file not found, skipping log persistence")
    except Exception as log_err:
        print(f"Warning: failed to persist logs to lakehouse: {log_err}")

    # Run DQ tests if enabled — always triggered, even when dlt job failed
    if run_dq and dq_repo_url:
        try:
            print(f"Running DQ tests for source: {source_name}")
            notebookutils.notebook.run(
                "generic_dbt_transform_v1",
                7200,
                {
                    "command": "test",
                    "models": f"tag:{source_name}",
                    "project_dir": f"/tmp/dbt_dq_{source_name}",
                    "dbt_subdir": ".dq",
                    "repo_url": dq_repo_url,
                    "repo_branch": dq_repo_branch,
                    "github_token": github_token,
                    "vault_url": vault_url,
                    "lakehouse_name": target.get("lakehouse", ""),
                    "lakehouse_id": target.get("lakehouse_id", ""),
                    "schema_name": target.get("schema_name", "dbo"),
                    "workspace_id": target.get("workspace_id", ""),
                    "useRootDefaultLakehouse": True,
                },
            )
            print("DQ tests completed successfully")
        except Exception as dq_err:
            print(f"Warning: DQ tests failed: {dq_err}")

